### Atividade 4 - NER - Ciências de Dados
### Professor André Camara
#### Alunos Diego Delgado, Laís de Jesus, Thiago Moura

A base de dados escolhida foi os dados de Televisão (tv.ibyte.json)

A proposta desse trabalho é utilizar o campo 'name' que se refere ao titulo na base de dados, mas após uma avaliação dos dados foi observado que alguns registros estavam muito resumidos, e que o campo 'url' estava mais completo. Então a primeira parte deste projeto é extrair o slug (parte final da URL) dos registros que estão na base de dados


In [1]:
import json
import re
import os
from urllib.parse import unquote

In [2]:
def clean_slug_url(url: str) -> str:
    if not url:
        return ""

    url_cleaned = url.split("?")[0].rstrip("/")

    slug = url_cleaned.split("/")[-1]
    if slug == "p":
        slug = url_cleaned.split("/")[-2]

    text_decoded = unquote(slug)
    
    title = text_decoded.replace("-", " ")
    title = re.sub(r"\s+", " ", title).strip()

    return title

In [3]:
def process_json_file(input_path: str, output_path: str):
    extracted_titles = []

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        for item in data:
            if isinstance(item, dict):
                microdata_list = item.get("microdata", [])

                for md_entry in microdata_list:
                    if isinstance(md_entry, dict) and md_entry.get("@type") == "Product":
                        raw_url = md_entry.get("url", "")
                        title = clean_slug_url(raw_url)

                        if title:
                            extracted_titles.append(title)
            else:
                print(f"Warning: Expected a dictionary in the JSON array, but found type {type(item)}. Skipping: {item}")
    elif isinstance(data, dict):
        microdata_list = data.get("microdata", [])
        for md_entry in microdata_list:
            if isinstance(md_entry, dict) and md_entry.get("@type") == "Product":
                raw_url = md_entry.get("url", "")
                title = clean_slug_url(raw_url)
                if title:
                    extracted_titles.append(title)
    else:
        print(f"Error: The JSON file root is neither a list nor a dictionary. Found type: {type(data)}. Cannot process.")
        return

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as file_out:
        for t in extracted_titles:
            file_out.write(t + "\n")

    print(
        f"Sucesso! {len(extracted_titles)} títulos foram extraídos e salvos em: {output_path}"
    )


if __name__ == "__main__":
    INPUT_FILE = "../data/raw/tv.ibyte.json"
    OUTPUT_FILE = "../data/processed/titles_tv_cleaned.txt"

    process_json_file(INPUT_FILE, OUTPUT_FILE)


Sucesso! 818 títulos foram extraídos e salvos em: ../data/processed/titles_tv_cleaned.txt
